# Single-Bus Substation

The single-bus configuration is the simplest possible design, with the lowest cost and also the lowest reliability. All equipment and switching devices are connected to a single main bus, which is always energized, as shown below. This configuration is commonly used in small electric distribution substations and wind farm collectors due to its simplicity and lower cost. Bus faults and breaker failures will result in an outage or significant voltage deviation to all branches in the entire substation as all branches share a common bus that will propagate the problem to said branches

## CIM Representation

In CIM, all buses and junctions are represented by the ConnectivityNode class. If a node corresponds to a bus bar, then a BusBarSection object is appended to the ConnectivityNode, as shown below in Figure 2. The single-bus configuration only includes a single main bus, with all distribution feeders connected to said bus. Full node-breaker switch representation adds a set of one Breaker and two Disconnector objects for each feeder added to the substation.

![single-bus](../images/single_bus.png)

----

## SingleBusSubtation

* new_branch

In [1]:
# Import cimgraph modules
from cimgraph.models import FeederModel, NodeBreakerModel
from cimgraph.databases import RDFlibConnection, XMLFile
import cimgraph.utils as utils
import cimgraph.data_profile.cimhub_2023 as cim

In [2]:
# Set environment variables
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2023'

In [3]:
connection = XMLFile(filename='new_single_bus.xml')

In [4]:
from cimbuilder.substation_builder import SingleBusSubstation

In [5]:
sub_builder = SingleBusSubstation(connection=connection, name="single_bus_sub", base_voltage=115000)
substation = sub_builder.substation


In [6]:
# Import 13 bus model from XML file
ieee13_feeder = cim.Feeder(mRID = '49AD8E07-3BF9-A4E2-CB8F-C3722F837B62')
xml13 = XMLFile(filename='../../sample_models/ieee13.xml')
ieee13_network = FeederModel(connection=xml13, container=ieee13_feeder, distributed=False)

In [7]:
sub_builder.new_feeder(breaker_number= 10, feeder=ieee13_feeder, feeder_network=ieee13_network)


In [8]:

assets13_feeder = cim.Feeder(mRID = '5B816B93-7A5F-B64C-8460-47C17D6E4B0F')
xml13assets = XMLFile(filename='../../sample_models/ieee13_assets.xml')
assets13_network = FeederModel(connection=xml13assets, container=assets13_feeder, distributed=False)

In [9]:
sub_builder.new_feeder(breaker_number = 20, feeder=assets13_feeder, feeder_network=assets13_network)

In [10]:
sub_builder.network.pprint(cim.Substation)

[
    {
        "@id": "09120c85-1d5b-422d-bc1e-6cc0039e04c8",
        "@type": "Substation",
        "name": "single_bus_sub",
        "NormalEnergizedFeeder": [
            {
                "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
                "@type": "Feeder"
            },
            {
                "@id": "5b816b93-7a5f-b64c-8460-47c17d6e4b0f",
                "@type": "Feeder"
            }
        ]
    }
]


In [11]:
sub_builder.upload()

In [12]:
sub_builder.pprint(cim.Feeder)

[
    {
        "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
        "@type": "Feeder",
        "NormalEnergizingSubstation": {
            "@id": "09120c85-1d5b-422d-bc1e-6cc0039e04c8",
            "@type": "Substation"
        }
    },
    {
        "@id": "5b816b93-7a5f-b64c-8460-47c17d6e4b0f",
        "@type": "Feeder",
        "NormalEnergizingSubstation": {
            "@id": "09120c85-1d5b-422d-bc1e-6cc0039e04c8",
            "@type": "Substation"
        }
    }
]


In [13]:
sub_builder.write_json_ld(filename='new_single_bus.json')

## Round-Trip Test: Load and Read from Database

### Delete all old entries from database and load new models

In [14]:
from cimloader.uploaders import BlazegraphUploader
loader = BlazegraphUploader()
loader.drop_all()

CIMG_URL environment variable is not set. Using Blazegraph default


In [15]:
loader.upload_from_xml(filename='../../sample_models/ieee13.xml')
loader.upload_from_xml(filename='../../sample_models/ieee13_assets.xml')
loader.upload_from_xml(filename='new_single_bus.xml')

HTTP/1.1 100 Continue

HTTP/1.1 200 OK
Content-Type: application/xml;charset=iso-8859-1
Content-Length: 62
Server: Jetty(9.4.18.v20190429)

<?xml version="1.0"?><data modified="4760" milliseconds="27"/>HTTP/1.1 100 Continue

HTTP/1.1 200 OK
Content-Type: application/xml;charset=iso-8859-1
Content-Length: 62
Server: Jetty(9.4.18.v20190429)

<?xml version="1.0"?><data modified="2927" milliseconds="22"/>HTTP/1.1 100 Continue

HTTP/1.1 200 OK
Content-Type: application/xml;charset=iso-8859-1
Content-Length: 61
Server: Jetty(9.4.18.v20190429)

<?xml version="1.0"?><data modified="168" milliseconds="14"/>

In [16]:
# Connect to Blazegraph Database
from cimgraph.databases import BlazegraphConnection
blazegraph = BlazegraphConnection()
network = NodeBreakerModel(container=substation, connection=blazegraph, distributed = False)

CIMG_URL environment variable is not set. Using Blazegraph default


In [17]:
# Print substation info
network.get_all_edges(cim.Substation)
network.pprint(cim.Substation)

[
    {
        "@id": "09120c85-1d5b-422d-bc1e-6cc0039e04c8",
        "@type": "Substation",
        "name": "single_bus_sub",
        "ConnectivityNodes": [
            {
                "@id": "06a56321-109c-4e5c-87e3-0a0b1218dad0",
                "@type": "ConnectivityNode"
            },
            {
                "@id": "1894bffb-2653-431b-a3a6-8813deea288a",
                "@type": "ConnectivityNode"
            },
            {
                "@id": "29485bb2-69bc-4b21-bfd2-7df67c73d67d",
                "@type": "ConnectivityNode"
            },
            {
                "@id": "4927bcfc-2bfe-4cc4-b16e-97d5fd30322b",
                "@type": "ConnectivityNode"
            },
            {
                "@id": "4c8bf07e-1f8d-4cbb-af7a-4808c1d5ec92",
                "@type": "ConnectivityNode"
            }
        ],
        "Equipments": [
            {
                "@id": "b129db27-eadc-41c2-9ff6-1f5778c956c7",
                "@type": "Disconnector"
          

In [18]:
# Print feeder info
network.get_all_edges(cim.Feeder)
network.pprint(cim.Feeder)

[
    {
        "@id": "49ad8e07-3bf9-a4e2-cb8f-c3722f837b62",
        "@type": "Feeder",
        "name": "ieee13nodeckt",
        "Location": {
            "@id": "8e4e3c92-0b7a-4f74-8fd2-cc10f74e452f",
            "@type": "Location"
        },
        "ConnectivityNodes": [
            {
                "@id": "0dcc57af-f4fa-457d-bb24-2efda9865a1a",
                "@type": "ConnectivityNode"
            },
            {
                "@id": "0f1e28c3-6c44-4f88-b79c-2fdbca4487b2",
                "@type": "ConnectivityNode"
            },
            {
                "@id": "2a6dc4dd-d3dc-434d-a187-d2c58a0a72c8",
                "@type": "ConnectivityNode"
            },
            {
                "@id": "30be5988-de57-4e0c-ab08-50d5a13d2c1b",
                "@type": "ConnectivityNode"
            },
            {
                "@id": "421e99be-a834-4809-b924-84d88f634a45",
                "@type": "ConnectivityNode"
            },
            {
                "@id": "63df

In [19]:
# Print total load served by substation from both feeders
total_load = 0
network.get_all_edges(cim.EnergyConsumer)
for load in network.graph[cim.EnergyConsumer].values():
    total_load = total_load + float(load.p)

print(f'total load is {total_load/1000} kW')

total load is 6937.0 kW


In [20]:
utils.get_all_data(network)

In [22]:
utils.write_xml(network, 'single_bus_and_feeders.xml')